In [0]:
%scala
//Load latest snapshot

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import org.apache.sedona.spark.SedonaContext
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData


val mapper = new ObjectMapper()
    mapper.configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
      .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
      .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val versionsBuilder = LayerVersions.builder()

versionsBuilder.layer(14533, 41917772)

val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

val env = "prod"
val database = "data_recovery"

implicit val sparky = spark
SedonaContext.create(spark)
ConfigLoader.forEnvironment(env)
SparkHelper.init(database)

new LoadFreshSnapshotData().run()

In [0]:
%scala

import org.apache.spark.sql.functions._
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository


val aptDS = new OrbisElementRepository("14533").readAll

display(aptDS)

In [0]:
%scala
val countryTagKey = "metadata:country" 
val stateTagKey   = "addr:state:en-Latn"

val state_wise_list = aptDS
    // Filter: country = USA
    .filter(exists(col("tags"), t => t.getField("tagKey").getField("key") ===  countryTagKey 
    && t.getField("value") === "USA"
    ))
     // Filter: metadata:apa:improvement = yes
    .filter(exists(col("tags"), t =>  t.getField("tagKey").getField("key") === "metadata:apa:improvement" 
    && t.getField("value") ===  "yes"))            
    // Extract state value from tags array
    .withColumn ("state", filter(col("tags"), t => t.getField("tagKey").getField("key") ===  stateTagKey).getItem(0).getField("value"))

val state_wise_count = state_wise_list.groupBy("state")
    .count()
    .orderBy(desc("state"))

display(state_wise_count)

In [0]:
%scala

val filtered_state = state_wise_list.filter(col("state") === "QC")
display(filtered_state)